# 04 · Routing Recall ρ(q)

Compute ρ(q) curves for centroid router and oracle across k = 1…100.
Compare all three generators against LAION ground truth.

> Requires ground truth files: `data/laion_1m/base.fbin`, `data/laion_1m/query.fbin`, `data/laion_1m/gt.ibin`

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
import sys
sys.path.insert(0, str(Path.cwd().parent / "src"))
from utils import read_fbin, read_ibin, balanced_kmeans
from metrics import rho_curve

Path("../results").mkdir(parents=True, exist_ok=True)

## Load LAION 1M base, queries and ground truth

In [ ]:
base_laion = read_fbin("../data/laion_1m/base.fbin")
queries    = read_fbin("../data/laion_1m/query.fbin")
gt         = read_ibin("../data/laion_1m/gt.ibin")
print(f"Base: {base_laion.shape}, queries: {queries.shape}, GT: {gt.shape}")

## Balanced k-means shard labels (N = 16)

In [ ]:
labels_laion = balanced_kmeans(base_laion, n_shards=16, seed=0)
print(f"Shard sizes: {np.bincount(labels_laion)}")

## ρ(q) for LAION 1M

In [ ]:
K_VALUES = [1, 2, 4, 8, 16, 32, 64, 100]
rho_laion = rho_curve(queries, base_laion, gt, labels_laion, k_values=K_VALUES)
print("Centroid ρ:", [f"{v:.3f}" for v in rho_laion["centroid"]])
print("Oracle  ρ:", [f"{v:.3f}" for v in rho_laion["oracle"]])

## ρ(q) for synthetic datasets

> TODO: load synthetic base vectors, compute ground truth with Faiss brute-force (k=200), then call `rho_curve`.
>
> Expected paths after notebook 03 runs:
> - `data/synth/perturbation_1m.fbin`
> - `data/synth/gmm_1m.fbin`
> - (cuVS results symlinked or copied from thesis repo)

In [ ]:
# TODO: for each synthetic dataset:
#   base_synth = read_fbin("../data/synth/<name>_1m.fbin")
#   gt_synth   = compute_gt_faiss(base_synth, queries_synth, k=200)
#   labels_synth = balanced_kmeans(base_synth, n_shards=16, seed=0)
#   rho_synth = rho_curve(queries_synth, base_synth, gt_synth, labels_synth)
pass

## Plot — centroid router and oracle side by side

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4), sharey=False)

for ax, router in zip(axes, ["centroid", "oracle"]):
    ax.plot(K_VALUES, rho_laion[router], label="LAION 1M", marker="o", color="black")
    # TODO: add synthetic curves
    # ax.plot(K_VALUES, rho_perturb[router], label="Perturbation", marker="s")
    # ax.plot(K_VALUES, rho_gmm[router],     label="GMM-diag",     marker="^")
    # ax.plot(K_VALUES, rho_cuvs[router],    label="cuVS+blur",    marker="D")
    ax.axhline(0.0, color="gray", linewidth=0.5)
    ax.set_xlabel("k")
    ax.set_ylabel("ρ̄")
    ax.set_title(f"{router.capitalize()} router")
    ax.legend()

plt.tight_layout()
plt.savefig("../results/04_rho_curves.png", dpi=120)
plt.show()